---
title: "Visualize"
subtitle: "From Excel to Python"
execute:
  echo: true
---

**In this lesson**

- One chart, one question: the test every chart has to pass
- `eda_toolkit`: one column, three views, with the mean and median drawn in
- Histograms on a log axis, and why the average hid the shape
- Boxes for comparing groups; stacked bars for a two-way table
- Bars, lines, and the axis-at-zero rule, revisited with code
- Small multiples: the same chart repeated, one panel per group
- A heatmap for two dimensions at once
- Fixing a bad chart, step by step
- `matplotlib`, `seaborn`, and `eda_toolkit`: which layer does what

**The one idea**

> A chart is an argument with a picture in it. Before drawing one, say the sentence it is supposed to make a reader believe. If you cannot, there is no chart to draw yet. If you can, the sentence tells you the chart type, the axis, and what to leave out.

::: {.content-visible when-format="html"}
[Download this notebook](chapter_06_lecture.ipynb){download="chapter_06_lecture.ipynb"} &nbsp;|&nbsp; [Open in Colab](https://colab.research.google.com/github/lshpaner/excel_to_python_exercises/blob/main/notebooks/chapter_06/chapter_06_lecture.ipynb){.colab-link}
:::

::: {.content-visible unless-format="html"}
Notebook and exercises for this chapter: `github.com/lshpaner/excel_to_python_exercises`, folder `notebooks/chapter_06/`.
:::

## Back to the decision

Five chapters in, the board has been shown four charts: trips per day, trips by hour, trips per weekday, and trips per station by region. Each one was chosen for a reason given at the time, but the reasons were scattered. This chapter collects them into a method, adds the two chart types that were missing (a histogram, which shows the shape of a column, and small multiples, which compare groups without stacking lines on top of each other), and ends with the chapter 10 question: which single chart would you put on the first page of the memo?

In [ ]:
try:
    import eda_toolkit
except ImportError:                      # Colab does not have it yet; local setups install it from requirements.txt
    %pip install -q eda_toolkit
    import eda_toolkit

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import StrMethodFormatter, PercentFormatter
from eda_toolkit import plot_distributions, data_doctor, box_violin_plot, stacked_crosstab_plot

LOCAL = "../../data/raw/"
URL = ("https://github.com/lshpaner/excel_to_python_exercises/"
       "releases/download/data/")
src = LOCAL if os.path.exists(LOCAL + "metro_trips.csv") else URL

trips = pd.read_csv(src + "metro_trips.csv")
trips["start_time"] = pd.to_datetime(trips["start_time"])
trips["hour"] = trips["start_time"].dt.hour
trips["weekday"] = trips["start_time"].dt.strftime("%a")
trips["date"] = trips["start_time"].dt.date
keep = (trips["duration"] > 1) & (trips["duration"] < 1440)      # chapter 3's keep column
trips = trips[keep].copy()

BLUE = "#1F4E79"
DAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
len(trips)

The setup has three new pieces. `eda_toolkit` is a library of exploratory-analysis functions (its author also wrote this book); the `try`/`except` installs it when it is missing, which is the case on a fresh Colab. The `keep` mask from chapter 3 is applied once at the top, so every chart in this chapter is drawn on clean rows; a chart of raw rows would be a chart of undocked bikes. And `plt.rcParams` sets defaults once (figure size, no top or right spine) so they do not have to be repeated in every cell. Set your defaults at the top of a notebook; a reader should not have to wonder why one chart looks different from the rest.

## One chart, one question

Every chart in this book was drawn to make a reader believe one sentence:

| Chapter | Sentence | Chart |
|---|---|---|
| 1 | Ridership runs about 1,000 a day and dips on holidays | Line, trips per day |
| 2 | Riding peaks at the commute hours | Line, trips by hour |
| 2 | Weekends are the busiest days per day | Bars, trips per weekday |
| 3 | Per station, the Westside out-rides downtown | Horizontal bars, trips per station |

The sentence came first each time, and it chose the chart. A line because hours and days are ordered and the shape between points means something. Bars because weekdays and regions are categories, read one at a time. Horizontal bars because region names are long and the comparison is of lengths.

The rule that follows: if a chart needs a paragraph to explain what it shows, it is making more than one argument, and it should be two charts.

## One column, three views

Before choosing a chart, look at the column. `data_doctor` draws the three standard views of one numeric column side by side, a density curve, a histogram, and a box plot, and lets you apply cutoffs to see what the column looks like inside a range. It also prints the quartiles and the outlier bounds that the box plot uses.

In [ ]:
#| label: fig-doctor
#| fig-cap: "data_doctor on duration with cutoffs at 2 and 180 minutes: density, histogram and box plot of the same column. The cutoff clips rather than drops, so every trip longer than 180 minutes is piled into the spike at the right edge of the histogram. The box plot's whisker ends at 57 minutes; everything past it is an outlier by the 1.5 IQR rule."
data_doctor(trips, "duration", apply_cutoff=True, lower_cutoff=2, upper_cutoff=180,
            plot_type="all", figsize=(14, 4), label_fontsize=11, tick_fontsize=9)

Note what the cutoff did: `apply_cutoff` clips values to the range instead of removing rows, so the 1,300 or so trips longer than 180 minutes are stacked at 180. That spike is the tail, folded. It is a reasonable way to keep the long trips in the count without letting them stretch the axis, as long as the caption says so.

All three views say the same thing in different vocabularies. The density curve peaks near 10 minutes and has a long right tail. The histogram shows the counts behind the curve. The box plot summarizes it in five numbers: the box is the middle half of trips, 7 to 27 minutes, the line inside is the median, and the dots past the whisker are the trips the box plot calls outliers, which here is every trip over 57 minutes. One in ten trips is an outlier by that rule, which tells you the rule was written for bell-shaped data and this column is not bell-shaped.

## Histograms: the shape of a column

Chapter 1 reported an average trip of 33 minutes and a median of 14 and said the gap was a clue. A histogram with both lines drawn on it is the picture of that clue. `plot_distributions` draws one per column, with the mean and median as vertical lines, and `log_scale_vars` switches a column to a log axis.

In [ ]:
#| label: fig-hist
#| fig-cap: "Trip duration, clean rows, with the mean (grey) and median (orange) drawn in. Left: linear axis, cut at 180 minutes. Right: the whole column on a log axis, where each tick is ten times the last, which is the only way to see the shape of a column that runs from 2 minutes to 24 hours."
#| layout-ncol: 2
plot_distributions(trips[trips["duration"] <= 180], ["duration"], plot_type="hist", density_function=None, stat="count",
                   bins=36, plot_mean=True, plot_median=True, mean_color="#7F7F7F", median_color="#C55A11",
                   hist_color=BLUE, hist_edgecolor=BLUE, y_axis_label="Trips", figsize=(5.5, 3.8),
                   custom_titles={"duration": "Duration, trips of 180 minutes or less"},
                   custom_xlabels={"duration": "minutes"}, label_fontsize=11, tick_fontsize=9)
plot_distributions(trips, ["duration"], plot_type="hist", density_function=None, stat="count",
                   log_scale_vars=["duration"], bins=36, plot_mean=True, plot_median=True,
                   mean_color="#7F7F7F", median_color="#C55A11", hist_color=BLUE, hist_edgecolor=BLUE,
                   y_axis_label="Trips", figsize=(5.5, 3.8), custom_titles={"duration": "Duration, all clean trips, log axis"},
                   custom_xlabels={"duration": "minutes (log scale)"}, label_fontsize=11, tick_fontsize=9)

Read the left panel: most trips are under 15 minutes, the count falls off fast, and a long thin tail runs to the right. Even with the trips past 180 minutes removed, the mean sits to the right of the median, pulled there by the tail; on the full column (right panel) the gap is twice as wide, 29 against 15. The median sits where the trips are. That is the whole chapter 1 argument in one picture, and it is why the median was the number to report.

The right panel is the same column on a log scale. Columns that span three orders of magnitude (2 minutes to 1,440) are unreadable on a linear axis and often roughly bell-shaped on a log one. Durations, incomes, file sizes, waiting times: if the mean is far above the median, try the log scale first.

Two choices every histogram forces. **Bin width**: 36 bins here, chosen so the median is visible and the tail is not a single bar. Too wide and the shape vanishes; too narrow and the chart is noise. Try three widths and keep the one that tells the story without inventing one. **Where to cut**: the linear panel is cut at 180 minutes and the title says so; drawn to the cap at 1,440 it is one bar at the left and a flat line, because the few hundred trips past 180 minutes spread across most of the axis. Cutting is fine; cutting silently is not.

## Comparing groups: boxes and stacked bars

A histogram per group is four charts. A box plot per group is one chart with four boxes, and when the question is "do these groups differ?" it is usually the better tool. `box_violin_plot` draws one box per category of a comparison column.

In [ ]:
#| label: fig-boxes
#| fig-cap: "Trip duration by pass type as box plots, outliers hidden. The two pass-holder boxes sit low and tight, with medians near 10 minutes; the two pay-per-ride boxes have medians near 30 and are three to four times as tall. The whole distribution shifts, not just the average."
box_violin_plot(trips, metrics_list=["duration"], metrics_comp=["passholder_type"], plot_type="boxplot",
                show_plot="subplots", subplot_figsize=(9, 4), showfliers=False, label_fontsize=11, tick_fontsize=10,
                custom_order=["Monthly Pass", "Annual Pass", "One Day Pass", "Walk-up"], ylim=(0, 140))

`showfliers=False` hides the outlier dots, which on this column would stack into a solid bar above every box and hide the boxes themselves. Say in the caption when you do that, as this one does.

Chapter 2's two-way table, pass type by bike type, was a `pivot_table` in chapter 5. As a chart it is a stacked bar, and the normalized version beneath it is the row-share table drawn: every bar the same height, the split inside it the comparison. `stacked_crosstab_plot` draws both from the raw frame.

In [ ]:
#| label: fig-crosstab
#| fig-cap: "Pass type by bike type as stacked bars: counts on top, each bar normalized to 100% beneath. The lower panel is chapter 2's share table as a picture: Monthly Pass holders are the only group that chooses electric more often than not."
common = dict(col="passholder_type", func_col=["bike_type"], legend_labels_list=[["electric", "standard"]],
              title=["bike type"], output="plots_only", color=[BLUE, "#9DB4CC"], width=0.6,
              custom_order=["Monthly Pass", "Annual Pass", "One Day Pass", "Walk-up"], label_fontsize=11, tick_fontsize=10)
stacked_crosstab_plot(trips, plot_type="regular", figsize=(10, 3.5), legend_loc="upper center", **common)
stacked_crosstab_plot(trips, plot_type="normalized", figsize=(10, 3.5), show_legend=False, show_values=True, **common)

The top panel answers "how many" and the bottom "what share", and chapter 2's warning applies to the pair: the counts alone make Monthly Pass look like the electric group because it is the biggest group of everything. Only the normalized panel is a fair comparison.

## Bars and lines, with code

Chapter 2's rule: a line when the x-axis is ordered and the gaps mean something, bars when the x-axis is a set of categories, and a bar's axis starts at zero because its length is what you read. Here is the per-weekday chart from chapter 2, then the same numbers drawn wrong on purpose.

In [ ]:
#| label: fig-bars-right-wrong
#| fig-cap: "Trips per day by weekday, clean rows. Left: axis at zero. Right: axis starting at 850, which turns a 15% difference into a cliff. Both are the same seven numbers."
n_days = pd.Series(pd.date_range("2026-01-01", "2026-03-31").strftime("%a")).value_counts()
per_day = (trips.groupby("weekday").size() / n_days).reindex(DAYS)

fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.6))
for ax, title in ((a, "Axis at zero"), (b, "Axis at 850")):
    ax.bar(DAYS, per_day.values, color=BLUE)
    ax.set_ylabel("Trips per day"); ax.set_title(title, loc="left", fontsize=11)
    ax.yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
a.set_ylim(0, per_day.max() * 1.15)
b.set_ylim(850, per_day.max() * 1.02)
plt.tight_layout()

The right panel is the most common chart lie in business reporting, and the tools make it easy: Excel and matplotlib both pick a "nice" axis range automatically, and for bars that range is often not zero. `set_ylim(0, ...)` is the line that makes a bar chart honest. Lines are different: a line's axis can start where the data starts, because a line is read by its slope, not its height.

## Small multiples

Chapter 2 found that pass holders and walk-up riders ride at different hours, and chapter 5 drew both groups' shares on one chart. Two lines on one axis is fine. Four is a tangle. The fix is small multiples: the same chart, drawn once per group, in a grid with shared axes so the eye compares panels without re-reading the scale.

In [ ]:
#| label: fig-small-multiples
#| fig-cap: "Share of each group's trips by hour of day, one panel per pass type. Shared axes make the panels comparable: the pass-holder panels carry a morning shoulder and an evening peak, the pay-per-ride panels are one afternoon hump."
groups = ["Monthly Pass", "Annual Pass", "One Day Pass", "Walk-up"]
by = pd.pivot_table(trips, index="hour", columns="passholder_type", values="trip_id", aggfunc="size")
share = by.div(by.sum(axis=0), axis=1)

fig, axes = plt.subplots(2, 2, figsize=(10, 5.6), sharex=True, sharey=True)
for ax, g in zip(axes.flat, groups):
    ax.fill_between(share.index, share[g], color=BLUE, alpha=0.25)
    ax.plot(share.index, share[g], color=BLUE, lw=2)
    ax.set_title(f"{g}  (n = {by[g].sum():,})", loc="left", fontsize=11)
    ax.set_xticks(range(0, 24, 6))
    ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
for ax in axes[1]:
    ax.set_xlabel("Hour of day")
for ax in axes[:, 0]:
    ax.set_ylabel("Share of group's trips")
plt.tight_layout()

`subplots(2, 2, sharex=True, sharey=True)` is the technique: a grid of axes, one loop, the same code in every panel. The reader's job becomes "find the panel that looks different," and here it is immediate. Annual Pass has a sharp peak at 8 and another at 17; Monthly Pass has a morning shoulder that climbs all day to a 17:00 peak; One Day Pass and Walk-up are a single afternoon hump that peaks around 14:00 to 16:00 with no morning at all. Pass holders ride to work, pay-per-ride customers ride in the afternoon. That is a cleaner statement of the two-populations finding than any table in chapter 2, and it also corrects it: the Monthly Pass commute is an evening commute more than a morning one.

Small multiples are the right tool whenever the question is "does this pattern hold in every group?" The groups can be pass types, regions, months, or stations. The limit is about twelve panels; past that, the eye stops comparing.

## A heatmap for two dimensions

Hour and weekday together is a 24 by 7 table, too many numbers to read as a table and too many lines to draw. A heatmap colors each cell by its value. `seaborn` draws one from a pivot table in one call.

In [ ]:
#| label: fig-heatmap
#| fig-cap: "Trips by hour and weekday, clean rows. The two commute bands run Monday to Friday; the weekend is a single broad afternoon."
hw = (pd.pivot_table(trips, index="weekday", columns="hour", values="trip_id", aggfunc="size")
        .reindex(DAYS))
fig, ax = plt.subplots(figsize=(10, 3.4))
sns.heatmap(hw, cmap="Blues", cbar_kws={"label": "Trips in the quarter", "format": StrMethodFormatter("{x:,.0f}")},
            linewidths=0.3, linecolor="white", ax=ax)
ax.set_xlabel("Hour of day"); ax.set_ylabel("")
ax.set_xticks([i + 0.5 for i in range(0, 24, 3)]); ax.set_xticklabels(range(0, 24, 3), rotation=0)
plt.tight_layout()

The heatmap shows what neither the by-hour line nor the by-weekday bars could: the weekday pattern, a morning band around 8 and a darker evening band at 16 to 17, stops dead on Saturday, and the weekend has one broad peak from noon to 17:00 with no morning at all. The single darkest cell in the quarter is Saturday at 15:00. Chapter 2's "two populations" is also "two weeks": a commuter week and a leisure week, sharing the same bikes.

Heatmaps have one rule: the color scale must start at zero or say why it does not, for the same reason as a bar's axis. `cmap="Blues"` is a sequential scale, light to dark, for counts. Use a diverging scale (two colors meeting at a neutral middle) only when the data have a meaningful center, like a difference from last quarter.

## Fixing a bad chart

Here is a chart of the kind that lands in inboxes every day, then three fixes. The data are average minutes by pass type and bike type from chapter 2.

In [ ]:
#| label: fig-bad-good
#| fig-cap: "The same eight numbers twice. Left: a default grouped bar chart with every default left in. Right: sorted, labeled, one color per bike type, axis at zero, the point in the title."
avg = pd.pivot_table(trips, index="passholder_type", columns="bike_type", values="duration", aggfunc="mean")
fig, (a, b) = plt.subplots(1, 2, figsize=(10.5, 3.8))

avg.plot(kind="bar", ax=a)
a.set_title("duration by passholder_type and bike_type")

order = avg.mean(axis=1).sort_values().index
avg.loc[order].plot(kind="barh", ax=b, color={"standard": "#9DB4CC", "electric": BLUE}, width=0.75)
b.set_xlim(0, avg.max().max() * 1.15); b.set_xlabel("Average minutes per trip"); b.set_ylabel("")
b.set_title("Pay-per-ride customers take long trips, on either bike", loc="left", fontsize=11)
b.legend(title="", frameon=False, loc="lower right")
for container in b.containers:
    b.bar_label(container, fmt="%.0f", padding=3, fontsize=9)
plt.tight_layout()

What changed, in the order it matters:

1. **The title states the finding**, not the column names. A reader who sees only the title should get the point.
2. **Sorted**, so the eye reads a ranking instead of hunting. Alphabetical order is nobody's question.
3. **Horizontal**, because the category labels are words, and words read left to right.
4. **Two colors with meaning**, a dark and a light of the same hue, instead of the default palette. Color is for one distinction; here it is bike type.
5. **Values on the bars**, so the chart doubles as the table and no one has to read a gridline.
6. **No rotated tick labels, no frame, no legend title.** Each was a default; each cost attention and paid none back.

None of this is taste. Each fix removes a step between the reader's eye and the sentence the chart exists to make.

## `matplotlib`, `seaborn`, and `eda_toolkit`

Three libraries appeared in this chapter, and they are layers. `matplotlib` is the foundation: every chart in this book is a matplotlib figure, and `ax.bar`, `ax.plot`, `ax.hist`, `set_ylim`, `set_title` are its vocabulary. pandas' `.plot()` is a thin layer over it. `seaborn` sits on top and adds chart types that take many lines in raw matplotlib (heatmaps, paired grids) and a better default style. `eda_toolkit` sits on top of both and packages the charts an analyst draws on every new dataset, a column's three views, a histogram with its mean and median, boxes by group, stacked crosstabs, so each is one call with the conventions already right (thousands separators, percent axes, a consistent order of categories).

The working rule: reach for `eda_toolkit` first when the chart is a standard one, seaborn or pandas `.plot()` for the rest, and matplotlib to adjust whatever comes back. The makeover section was pure matplotlib on purpose: the fixes are the lesson, and you need to know how to make them by hand to recognize when a library has made them for you.

## Back to the decision

The board gets one chart on the first page of the memo. Of the charts in this chapter, the heatmap is the one that says the most in one look: two commute bands and a leisure weekend, which is the whole "two populations" argument and the reason any single policy will fit neither. The small multiples are the supporting exhibit. The histogram is for the appendix, where the median-not-mean caveat lives.

That choice is chapter 10's problem, but the method for making it is this chapter's: write the sentence, pick the chart that makes a reader believe it fastest, remove everything else.

## Looking ahead

- **Worked exercise:** `chapter_06_guided.ipynb`. `data_doctor` on a second column, duration by pass type with `box_violin_plot` as violins, pass type by weekday with `stacked_crosstab_plot`; then the bad-to-good makeover applied to a chart of trips per station by region.
- **On your own:** `chapter_06_solo.ipynb`. Build the heatmap for Walk-up riders only, draw the trips-per-day line with weekends shaded, and write the one sentence each chart is for. Then pick one of the two for a first page and say why.
- **Before the next chapter:** chapter 7 asks whether the difference between weekday and weekend trip lengths is real or luck. Look at the small multiples again and guess.

## Appendix: reproducing this chapter {.unnumbered}

`chapter_06_lecture.ipynb` is generated from this page; exercises come from `tools/build_chapter_06_exercises.py`. All charts are drawn on clean rows (the chapter 3 `keep` mask) from `data/raw/`, and every figure in the book is written as SVG. `eda_toolkit` is on PyPI (`pip install eda_toolkit`) and in `requirements.txt`.